# PyTorchにおけるCPUとGPU（CUDA）の使い方

---
## 目的
PyTorch (`torch`) において，CPUとGPU（CUDA）を切り替えて計算を行う方法について学ぶ．

## CPUとGPUについて

ディープラーニングの学習・推論では，大量の行列演算（畳み込みや行列積など）を繰り返し計算する必要があります．
CPUは少数のコアで様々な処理を順番にこなすことが得意な一方，GPU（Graphics Processing Unit）は数千個の小さなコアを持ち，同じ種類の演算を大量に並列実行することが得意です．そのため，ディープラーニングの計算はGPUを利用することで大幅に高速化できます．

PyTorchでは，NVIDIA製GPU上での並列計算を行うためのプラットフォームである **CUDA** を通じて，GPUを利用した計算を行うことができます．PyTorchのプログラムでは，Tensorやモデル（`nn.Module`）が「どのデバイス（CPU or GPU）上に存在するか」を意識してプログラムを記述する必要があります．

以下では，PyTorchにおけるCPUとGPUの基本的な扱い方を，プログラムを交えて説明します．

## GPUが利用可能かを確認する

`torch.cuda.is_available()`を用いることで，実行環境でGPU（CUDA）が利用可能かどうかを確認することができます．
`True`が返された場合はGPUを利用した計算が可能であり，`False`の場合はCPUのみが利用可能です．

（Google Colab.でGPUを使用する場合は，上部のメニューバーの「ランタイム」→「ランタイムのタイプを変更」からハードウェアアクセラレータをGPUにしてください．）

In [ ]:
import torch

print('GPU (CUDA) が利用可能か:', torch.cuda.is_available())

### GPUの詳細情報を確認する

GPUが利用可能な場合，`torch.cuda`以下の関数を用いて，利用可能なGPUの数や名前などの詳細情報を確認することができます．

* `torch.cuda.device_count()`：利用可能なGPUの数
* `torch.cuda.current_device()`：現在使用されているGPUの番号
* `torch.cuda.get_device_name(番号)`：指定した番号のGPUの名前

In [ ]:
if torch.cuda.is_available():
    print('利用可能なGPUの数:', torch.cuda.device_count())
    print('現在使用中のGPU番号:', torch.cuda.current_device())
    print('GPUの名前:', torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print('GPUが利用できないため，詳細情報は表示できません．')

また，Colab.上では`!nvidia-smi`コマンドを実行することで，GPUの型番やメモリ使用量など，より詳細な情報を確認することもできます．

In [ ]:
!nvidia-smi

## Tensorのデバイスを指定する（`.cpu()`, `.cuda()`）

PyTorchで作成したTensorは，デフォルトではCPU上に配置されます．
Tensorが現在どのデバイス上にあるかは`.device`属性で確認できます．

Tensorを別のデバイスへ転送するには，以下のメソッドを使用します．

* `.cuda()`：TensorをGPUへ転送する
* `.cpu()`：TensorをCPUへ転送する

ただし，`.cuda()`はGPUが存在しない環境で実行するとエラーになってしまう点に注意が必要です．

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
print('作成直後のデバイス:', x.device)  # デフォルトはCPU

if torch.cuda.is_available():
    x_gpu = x.cuda()             # GPUへ転送
    print('GPUへ転送後のデバイス:', x_gpu.device)

    x_cpu = x_gpu.cpu()          # 再びCPUへ転送
    print('CPUへ転送後のデバイス:', x_cpu.device)
else:
    print('GPUが利用できないため，.cuda()による転送はスキップします．')

## `device`と`.to()`によるデバイス指定（現在の主流）

上記の`.cpu()`, `.cuda()`は直感的ですが，GPUが存在しない環境で`.cuda()`を実行するとエラーになってしまうため，CPU環境とGPU環境の両方で同じプログラムを動かしたい場合には不便です．

そこで現在では，`torch.device`でデバイスを表すオブジェクトを作成し，`.to(device)`メソッドで転送先を指定する方法が主流となっています．
この方法では，`torch.cuda.is_available()`の結果に応じて`device`の中身を切り替えることで，**GPUの有無によらず同じプログラムをそのまま実行できる（デバイスに依存しないコードが書ける）**という利点があります．

In [ ]:
# 利用可能であればGPU，そうでなければCPUを使用するdeviceを作成
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用するデバイス:', device)

x = torch.tensor([1.0, 2.0, 3.0])
x = x.to(device)  # deviceに応じてCPU/GPUへ転送
print('転送後のデバイス:', x.device)

# dtypeを同時に指定することも可能
x = x.to(device, dtype=torch.float64)
print(x.dtype)

## 複数のGPUがある場合のデバイス指定

GPUを複数搭載した環境では，先ほど確認した`torch.cuda.device_count()`が2以上の値になります．
PyTorchでは，搭載されている各GPUに対して`0`番から始まるインデックス番号が割り振られており，`cuda:0`, `cuda:1`, ... のように番号を指定することで，計算に使用するGPUを明示的に選択することができます．

GPUを番号で指定する方法には，主に以下のようなものがあります．

* `torch.device('cuda:番号')`を作成し，`.to()`で転送する（あるいはTensor作成時に`device=`引数へ直接指定する）
* `.cuda(番号)`のように，`.cuda()`メソッドの引数に番号を指定する
* `torch.cuda.set_device(番号)`で，以降のデフォルトで使用するGPUそのものを変更する

以下では，利用可能な全てのGPUに対して，それぞれのGPU上にTensorを作成する例を示します．（GPUが1枚しかない環境では，`cuda:0`のみが対象となります．）

In [ ]:
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_i = torch.device(f'cuda:{i}')
        x = torch.tensor([1.0, 2.0, 3.0], device=device_i)  # Tensor作成時にdeviceを直接指定することも可能
        print(f'GPU {i} ({torch.cuda.get_device_name(i)}) 上のTensor:', x.device)
else:
    print('GPUが利用できないため，この例は実行できません．')

既存のTensorやモデルに対しては，`.cuda(番号)`のように`.cuda()`へ直接番号を渡すことでも，特定のGPUへ転送できます．
また，`torch.cuda.set_device(番号)`を実行すると，それ以降`'cuda'`とだけ指定した場合に使用されるデフォルトのGPUを変更することができます．

In [ ]:
if torch.cuda.is_available():
    x = torch.tensor([1.0, 2.0, 3.0])
    x_gpu0 = x.cuda(0)  # 0番目のGPUへ転送（.to('cuda:0')と同じ）
    print('cuda(0)で転送したTensorのデバイス:', x_gpu0.device)

    # 以降，'cuda'とだけ指定した場合に使用されるデフォルトのGPUを変更する
    torch.cuda.set_device(0)
    print('デフォルトのGPU番号:', torch.cuda.current_device())
else:
    print('GPUが利用できないため，この例は実行できません．')

### 補足：使用するGPUをプロセス単位で制限する

複数人でGPUサーバを共有する場合など，特定のGPUだけをプログラムに使わせたいことがあります．その場合は，PyTorchのプログラムを実行する前に環境変数`CUDA_VISIBLE_DEVICES`を設定する方法がよく使われます．

```bash
CUDA_VISIBLE_DEVICES=1 python train.py
```

このように実行すると，物理的な2枚目のGPU（0番始まりなので番号は`1`）だけがそのプロセスから見えるようになり，プログラム内からは`cuda:0`としてアクセスできます．

### 補足：複数のGPUを使って1つのモデルを学習する

ここまでは「複数あるGPUの中からどれを使うか」を選択する方法を紹介しましたが，PyTorchには複数のGPUを同時に使い，1つのモデルの学習を高速化するための仕組みも用意されています．

* `torch.nn.DataParallel`：手軽に導入できるが，現在は`DistributedDataParallel`の使用が推奨されている
* `torch.nn.parallel.DistributedDataParallel`（DDP）：複数GPU・複数ノードでの学習に対応した，現在推奨されている方法

これらは本ノートブックの範囲を超える発展的な内容のため，詳細は[PyTorch公式のDistributed Data Parallelに関するドキュメント](https://pytorch.org/tutorials/beginner/ddp_series_theory.html)を参照してください．

## モデル（`nn.Module`）のデバイス指定

`.to(device)`は，Tensorだけでなく`nn.Module`を継承したモデルに対しても使用できます．
モデルに対して`.to(device)`を実行すると，モデルが持つ全てのパラメータ（重み）が指定したデバイスへ転送されます．

学習・推論を行う際は，**モデルと入力データを同じデバイスに揃える**必要があります．

In [ ]:
import torch.nn as nn

model = nn.Linear(3, 1)  # 入力3次元 -> 出力1次元の全結合層
model = model.to(device)  # モデルのパラメータをdeviceへ転送

x = torch.tensor([[1.0, 2.0, 3.0]])
x = x.to(device)  # 入力データも同じdeviceへ転送

y = model(x)
print(y)
print('モデルのパラメータのデバイス:', next(model.parameters()).device)
print('出力のデバイス:', y.device)

## 注意：デバイスの不一致によるエラー

モデルと入力データが異なるデバイス上にある場合，`RuntimeError`が発生します．
これはPyTorchを使う上で非常によく遭遇するエラーの一つです．

以下のプログラムでは，GPU上に配置したモデルに対して，CPU上のTensorを直接入力してエラーを発生させています．
（GPUが利用できない環境では，このセルは実行されません．）

In [ ]:
if torch.cuda.is_available():
    model_gpu = nn.Linear(3, 1).to('cuda')  # モデルをGPUへ
    x_cpu = torch.tensor([[1.0, 2.0, 3.0]])  # Tensorは何も指定しなければCPUのまま

    try:
        model_gpu(x_cpu)  # デバイスが異なるためエラーになる
    except RuntimeError as e:
        print('RuntimeErrorが発生しました:')
        print(e)
else:
    print('この例を実行するにはGPUランタイムが必要です．')

## まとめ

CPU・GPUの指定方法を以下の表にまとめます．

| 方法 | 説明 |
| --- | --- |
| `.cpu()` | TensorやモデルをCPUへ転送する |
| `.cuda()` | TensorやモデルをGPUへ転送する．GPUが存在しない環境で実行するとエラーになる |
| `.to(device)` | `device`の中身（`'cpu'`または`'cuda'`）に応じて柔軟に転送先を切り替えられる．**GPUの有無によらず同じコードが動作するため，現在はこちらの方法が主流** |
| `.to('cuda:番号')` / `.cuda(番号)` | 複数GPUがある環境で，使用する特定のGPUを番号で指定する |

実際にモデルを学習する際は，スクリプトの冒頭で

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
```

のように`device`を定義し，モデルと入力データの両方に対して`.to(device)`を適用するという書き方が一般的です．
GPUが複数ある環境では，`torch.device('cuda:0')`のように番号を含めて指定することで，使用するGPUを明示的に選択できます．